# Tahap 3: Case Retrieval (IndoBERT Embedding)
**CBR - Pidana Umum: Pemalsuan**

Tujuan: Temukan kasus lama yang paling mirip dengan query kasus baru.

**Pendekatan:** IndoBERT (`indobenchmark/indobert-base-p1`) untuk sentence embedding,
kemudian Cosine Similarity untuk retrieval.

**Langkah:**
1. Load data kasus dari Tahap 2
2. Bangun embedding IndoBERT untuk semua kasus
3. Implementasi fungsi `retrieve(query, k)`
4. Splitting data (train/test)
5. Pengujian awal dengan query uji

## 0. Instalasi Library

In [1]:
!pip install transformers torch scikit-learn numpy pandas tqdm


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Import & Konfigurasi

In [12]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

# Path
PROCESSED_DIR = Path("../data/processed")
EVAL_DIR      = Path("../data/eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Model IndoBERT
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 512   # Max token IndoBERT
BATCH_SIZE = 8     # Sesuaikan dengan RAM/GPU Anda (turunkan jika OOM)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
print(f"🤖 Model : {MODEL_NAME}")
print(f"📁 Data  : {PROCESSED_DIR.resolve()}")

🖥️  Device: cpu
🤖 Model : indobenchmark/indobert-base-p1
📁 Data  : E:\Semester 6\Penalaran Komputer\SubCPMK 3\Test_1\data\processed


## 2. Load Data Kasus

In [13]:
# Load dari JSON (berisi text_full)
json_path = PROCESSED_DIR / "cases.json"
with open(json_path, encoding="utf-8") as f:
    cases = json.load(f)

df = pd.DataFrame(cases)
print(f"✅ {len(df)} kasus dimuat")
print(f"   Kolom: {list(df.columns)}")

# Buat teks representasi untuk embedding:
# Gabungkan dakwaan + ringkasan_fakta + pasal untuk representasi yang lebih kaya
def build_embedding_text(row) -> str:
    """
    Bangun teks representasi kasus untuk di-embed.
    Prioritaskan konten kunci agar embedding lebih fokus.
    """
    parts = []
    if row.get("pasal") and row["pasal"] != "TIDAK_DITEMUKAN":
        parts.append(f"Pasal: {row['pasal']}")
    if row.get("dakwaan") and row["dakwaan"] != "TIDAK_DITEMUKAN":
        parts.append(f"Dakwaan: {row['dakwaan'][:300]}")
    if row.get("ringkasan_fakta") and row["ringkasan_fakta"] != "TIDAK_DITEMUKAN":
        parts.append(f"Fakta: {row['ringkasan_fakta'][:300]}")
    
    # Jika konten kunci tidak ditemukan, fallback ke text_full (dipotong)
    if not parts:
        return row.get("text_full", "")[:800]
    
    return " ".join(parts)

df["embedding_text"] = df.apply(build_embedding_text, axis=1)
print(f"\n📝 Contoh teks embedding (case_001):")
print(df["embedding_text"].iloc[0][:400])

✅ 35 kasus dimuat
   Kolom: ['case_id', 'original_filename', 'no_perkara', 'tanggal', 'jenis_perkara', 'pasal', 'terdakwa', 'penuntut', 'pengadilan', 'ringkasan_fakta', 'dakwaan', 'amar_putusan', 'vonis', 'word_count', 'sentence_count', 'unique_words', 'top_keywords', 'pemalsuan_keywords', 'text_full']

📝 Contoh teks embedding (case_001):
Pasal: 374, 253, 263, 263 1, 197 1 Dakwaan: bahwa terdakwa listyo herlambang,sh bin kalistiono telah membantu kepada setiyono, s.ag (dalam perkara lain) pada hari jum'at tanggal tanggal 25 januari 2013 sekitar pukul 10.35 wib atau setidak- tidaknya masih termasuk bulan januari 2013 atau setidak tidaknya pada suatu waktu dalam tahun 2013 berte Fakta: secara utuh dalam persidangan, sehingga dalam pu


## 3. Splitting Data (Train / Test = 80:20)

In [14]:
# Split berdasarkan case_id
all_case_ids = df["case_id"].tolist()

train_ids, test_ids = train_test_split(
    all_case_ids,
    test_size=0.2,
    random_state=42
)

df_train = df[df["case_id"].isin(train_ids)].reset_index(drop=True)
df_test  = df[df["case_id"].isin(test_ids)].reset_index(drop=True)

print(f"📊 Split Data:")
print(f"  Train : {len(df_train)} kasus ({len(df_train)/len(df)*100:.0f}%)")
print(f"  Test  : {len(df_test)} kasus ({len(df_test)/len(df)*100:.0f}%)")

# Simpan split info
split_info = {
    "train_ids": train_ids,
    "test_ids": test_ids,
    "split_ratio": "80:20"
}
with open(EVAL_DIR / "data_split.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)
print(f"✅ Split info disimpan di {EVAL_DIR}/data_split.json")

📊 Split Data:
  Train : 28 kasus (80%)
  Test  : 7 kasus (20%)
✅ Split info disimpan di ..\data\eval/data_split.json


## 4. Load Model IndoBERT

In [15]:
print(f"⏳ Memuat model IndoBERT: {MODEL_NAME}")
print("   (Proses ini mungkin memakan waktu beberapa menit pertama kali...)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model     = model.to(device)
model.eval()

print(f"✅ Model IndoBERT berhasil dimuat!")
print(f"   Parameter: {sum(p.numel() for p in model.parameters()):,}")

⏳ Memuat model IndoBERT: indobenchmark/indobert-base-p1
   (Proses ini mungkin memakan waktu beberapa menit pertama kali...)
✅ Model IndoBERT berhasil dimuat!
   Parameter: 124,441,344


## 5. Fungsi Embedding

In [16]:
def mean_pooling(model_output, attention_mask):
    """
    Mean Pooling: Rata-rata token embeddings dengan mempertimbangkan attention mask.
    Menghasilkan satu vektor per dokumen.
    """
    token_embeddings = model_output[0]  # (batch, seq_len, hidden_size)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / \
           torch.clamp(input_mask_expanded.sum(1), min=1e-9)


def get_embeddings(texts: list, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """
    Buat embedding untuk list of texts menggunakan IndoBERT.
    Mengembalikan numpy array shape (n_texts, hidden_size).
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i + batch_size]
        
        encoded = tokenizer(
            batch,
            max_length=MAX_LENGTH,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        
        with torch.no_grad():
            output = model(**encoded)
        
        embeddings = mean_pooling(output, encoded["attention_mask"])
        
        # Normalisasi L2 (untuk cosine similarity yang lebih efisien)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())
    
    return np.vstack(all_embeddings)


print("✅ Fungsi embedding siap")

✅ Fungsi embedding siap


## 6. Bangun Case Vectors (Embedding Semua Kasus)

In [17]:
print("⏳ Membuat embedding untuk semua kasus train...")
print(f"   Jumlah kasus train: {len(df_train)}")

# Embed semua kasus train
train_texts = df_train["embedding_text"].tolist()
train_embeddings = get_embeddings(train_texts)

print(f"✅ Embedding selesai! Shape: {train_embeddings.shape}")

# Simpan embedding ke file numpy (agar tidak perlu hitung ulang)
np.save(EVAL_DIR / "train_embeddings.npy", train_embeddings)
train_case_ids = df_train["case_id"].tolist()
with open(EVAL_DIR / "train_case_ids.json", "w") as f:
    json.dump(train_case_ids, f)

print(f"💾 Embedding disimpan di {EVAL_DIR}/train_embeddings.npy")

⏳ Membuat embedding untuk semua kasus train...
   Jumlah kasus train: 28


Embedding: 100%|██████████| 4/4 [00:05<00:00,  1.47s/it]

✅ Embedding selesai! Shape: (28, 768)
💾 Embedding disimpan di ..\data\eval/train_embeddings.npy


## 7. Fungsi Retrieve

In [18]:
# Load data yang diperlukan untuk retrieve
case_lookup = {c["case_id"]: c for c in cases}  # Dict untuk akses cepat

def retrieve(query: str, k: int = 5, return_details: bool = False) -> list:
    """
    Fungsi utama retrieval: Temukan k kasus paling mirip dengan query.
    
    Args:
        query    : Teks query kasus baru (dakwaan/fakta/pasal)
        k        : Jumlah kasus teratas yang dikembalikan
        return_details: Jika True, kembalikan dict dengan skor similarity
    
    Returns:
        List case_id terurut dari paling mirip
    """
    # Step 1: Pre-process query (sudah lowercase dari tahap 1)
    query_clean = query.lower().strip()
    
    # Step 2: Hitung vektor query
    query_embedding = get_embeddings([query_clean])
    
    # Step 3: Hitung cosine similarity dengan semua case vectors
    similarities = cosine_similarity(query_embedding, train_embeddings)[0]
    
    # Step 4: Ambil top-k case_id
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_case_ids = [train_case_ids[i] for i in top_k_indices]
    top_k_scores   = [float(similarities[i]) for i in top_k_indices]
    
    if return_details:
        return [
            {
                "case_id": cid,
                "similarity": score,
                "no_perkara": case_lookup.get(cid, {}).get("no_perkara", "-"),
                "vonis": case_lookup.get(cid, {}).get("vonis", "-"),
                "amar_putusan": case_lookup.get(cid, {}).get("amar_putusan", "-")[:200]
            }
            for cid, score in zip(top_k_case_ids, top_k_scores)
        ]
    
    return top_k_case_ids


print("✅ Fungsi retrieve() siap!")

✅ Fungsi retrieve() siap!


## 8. Buat Query Uji (Ground Truth)

In [19]:
# Gunakan kasus test sebagai query uji (simulasi kasus baru)
# Ground truth: kasus test sebenarnya adalah kasus yang paling relevan

test_queries = []
for _, row in df_test.iterrows():
    # Query = teks embedding kasus test (mensimulasikan kasus baru)
    query_text = row["embedding_text"]
    
    # Ground truth: case_id itu sendiri (seharusnya masuk top-k jika ada di train)
    # Namun karena test tidak ada di train, kita cari case paling mirip di train
    # yang memiliki pasal sama sebagai ground truth
    pasal_query = row.get("pasal", "")
    
    # Cari kasus train dengan pasal sama sebagai ground truth
    similar_train = df_train[
        df_train["pasal"].str.contains(
            pasal_query[:10] if pasal_query != "TIDAK_DITEMUKAN" else "000",
            na=False, regex=False
        )
    ]["case_id"].tolist()
    
    test_queries.append({
        "query_id"   : f"q_{row['case_id']}",
        "query_text" : query_text[:500],
        "source_case": row["case_id"],
        "pasal"      : pasal_query,
        "ground_truth_case_ids": similar_train[:3]  # Top 3 ground truth
    })

# Simpan queries
with open(EVAL_DIR / "queries.json", "w", encoding="utf-8") as f:
    json.dump(test_queries, f, ensure_ascii=False, indent=2)

print(f"✅ {len(test_queries)} query uji disimpan di {EVAL_DIR}/queries.json")
print(f"\nContoh query:")
print(json.dumps(test_queries[0], ensure_ascii=False, indent=2)[:500])

✅ 7 query uji disimpan di ..\data\eval/queries.json

Contoh query:
{
  "query_id": "q_case_014",
  "query_text": "Pasal: 199, 263, 263 2, 199 1, 197 1 Dakwaan: dan bukan didasarkan pada tidak terbuktinya suatu unsur perbuatan yang didakwakan, atau apabila pembebasan itu sebenarnya adalah merupakan putusan lepas dari segala tuntutan hukum, atau apabila dalam menjatuhkan putusan itu pengadilan telah melampaui batas kewenangannya (meskipun hal ini tidak diaju Fakta: yang diungkapkan oleh pengadilan negeri tarakan sebagai berikut : 1.1.",
  "source_case": "case_014


## 9. Demo Retrieval

In [20]:
# Demo dengan query manual
demo_query = """terdakwa terbukti memalsukan surat keterangan resmi dan 
menggunakan dokumen palsu untuk kepentingan pribadi. 
didakwa melanggar pasal 263 kuhp tentang pemalsuan surat."""

print("🔍 Demo Retrieval")
print("=" * 60)
print(f"Query: {demo_query[:200]}")
print("="*60)

results = retrieve(demo_query, k=5, return_details=True)

print(f"\n📋 Top-5 Kasus Paling Mirip:")
for i, r in enumerate(results, 1):
    print(f"\n[{i}] Case ID   : {r['case_id']}")
    print(f"    No Perkara: {r['no_perkara']}")
    print(f"    Similarity: {r['similarity']:.4f}")
    print(f"    Vonis     : {r['vonis']}")
    print(f"    Amar      : {r['amar_putusan'][:150]}...")

🔍 Demo Retrieval
Query: terdakwa terbukti memalsukan surat keterangan resmi dan 
menggunakan dokumen palsu untuk kepentingan pribadi. 
didakwa melanggar pasal 263 kuhp tentang pemalsuan surat.


Embedding: 100%|██████████| 1/1 [00:00<00:00,  8.33it/s]


📋 Top-5 Kasus Paling Mirip:

[1] Case ID   : case_009
    No Perkara: 567/PDT/2018/PT
    Similarity: 0.6836
    Vonis     : TIDAK_DITEMUKAN
    Amar      : menolak gugatan penggugat (terdakwa); bahwa berita acara pemeriksaan laboratoris kriminalisitik no. lab; 3074/dtf/2019 tertanggal 2 september 2019 yan...

[2] Case ID   : case_015
    No Perkara: 75/PID/2024/
    Similarity: 0.6496
    Vonis     : TIDAK_DITEMUKAN
    Amar      : 3 (tiga) orang sebagai terpidana, yakni untuk terpidana hendra anak dari almarhum tiam seng dijatuhi pidana penjara selama 3 (tiga) tahun 4 (empat) bu...

[3] Case ID   : case_028
    No Perkara: 2/PID.B/2019/PN
    Similarity: 0.6417
    Vonis     : TIDAK_DITEMUKAN
    Amar      : sebagaimana yang akandisebutkan di bawah ini;menimbang bahwa sebelum menjatuhkan pidana mahkamah agungakan mempertimbangkan keadaan yang memberatkan d...

[4] Case ID   : case_002
    No Perkara: 1530/PID.B/2014/PN.BDG
    Similarity: 0.6392
    Vonis     : TIDAK_DITEMUKAN
    A

In [21]:
# Demo dengan kasus test pertama
if len(df_test) > 0:
    sample_test = df_test.iloc[0]
    print(f"\n🔍 Retrieval untuk kasus test: {sample_test['case_id']}")
    print(f"   Pasal: {sample_test['pasal']}")
    print(f"   Query: {sample_test['embedding_text'][:300]}")
    print()
    
    results_test = retrieve(sample_test["embedding_text"], k=5, return_details=True)
    print("Top-5 kasus mirip:")
    for r in results_test:
        print(f"  {r['case_id']} | sim={r['similarity']:.4f} | {r['no_perkara']} | {r['vonis']}")

print("\n✅ Tahap 3 selesai! Lanjutkan ke notebook 04_solution_reuse.ipynb")


🔍 Retrieval untuk kasus test: case_014
   Pasal: 199, 263, 263 2, 199 1, 197 1
   Query: Pasal: 199, 263, 263 2, 199 1, 197 1 Dakwaan: dan bukan didasarkan pada tidak terbuktinya suatu unsur perbuatan yang didakwakan, atau apabila pembebasan itu sebenarnya adalah merupakan putusan lepas dari segala tuntutan hukum, atau apabila dalam menjatuhkan putusan itu pengadilan telah melampaui bat



Embedding: 100%|██████████| 1/1 [00:00<00:00,  5.13it/s]

Top-5 kasus mirip:
  case_008 | sim=0.8643 | 16/PID.B/2012/ | lepas dari segala tuntutan
  case_001 | sim=0.8386 | 309/PID.B/2013/PN.PKL | TIDAK_DITEMUKAN
  case_002 | sim=0.8306 | 1530/PID.B/2014/PN.BDG | TIDAK_DITEMUKAN
  case_010 | sim=0.8272 | 66/PID.B/2015/PN.BAU | TIDAK_DITEMUKAN
  case_033 | sim=0.8247 | 2487/PID.B/2018/PN.TNG | TIDAK_DITEMUKAN

✅ Tahap 3 selesai! Lanjutkan ke notebook 04_solution_reuse.ipynb
